# Single Image Super Resolution using CNN and GAN
Minor Project



## 1. Imports and Environment Setup


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import cv2
import numpy as np
import matplotlib.pyplot as plt
import math

from torchvision.models import vgg19
from torchvision import transforms

## 2. Dataset Preparation (LR–HR Image Pairs)


In [ ]:
class SRDataset(Dataset):
    def __init__(self, image_paths, scale=4):
        self.image_paths = image_paths
        self.scale = scale

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        hr = cv2.imread(self.image_paths[idx])
        hr = cv2.cvtColor(hr, cv2.COLOR_BGR2RGB)

        h, w, _ = hr.shape
        lr = cv2.resize(hr, (w // self.scale, h // self.scale))
        lr = cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC)

        hr = hr / 255.0
        lr = lr / 255.0

        hr = torch.tensor(hr).permute(2, 0, 1).float()
        lr = torch.tensor(lr).permute(2, 0, 1).float()

        return lr, hr


## 3. SRCNN Model (Baseline CNN)


In [ ]:
class SRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 9, padding=4),
            nn.ReLU(),
            nn.Conv2d(64, 32, 5, padding=2),
            nn.ReLU(),
            nn.Conv2d(32, 3, 5, padding=2)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
image_paths = ["image1.jpg", "image2.jpg"]  # put 2–5 images here

dataset = SRDataset(image_paths)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

srcnn = SRCNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(srcnn.parameters(), lr=1e-4)

for epoch in range(5):
    total_loss = 0
    for lr, hr in loader:
        sr = srcnn(lr)
        loss = criterion(sr, hr)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"SRCNN Epoch {epoch+1}, Loss: {total_loss/len(loader)}")


## 4. SRGAN Model (Generative Super Resolution)


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.PReLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.initial = nn.Sequential(
            nn.Conv2d(3, 64, 9, padding=4),
            nn.PReLU()
        )
        self.residuals = nn.Sequential(
            *[ResidualBlock(64) for _ in range(5)]
        )
        self.upsample = nn.Sequential(
            nn.Conv2d(64, 256, 3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(64, 3, 9, padding=4)
        )

    def forward(self, x):
        x = self.initial(x)
        x = self.residuals(x)
        return self.upsample(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
G = Generator()
D = Discriminator()

content_loss = nn.MSELoss()
adversarial_loss = nn.BCELoss()

vgg = vgg19(pretrained=True).features[:36].eval()
for p in vgg.parameters():
    p.requires_grad = False

opt_G = optim.Adam(G.parameters(), lr=1e-4)
opt_D = optim.Adam(D.parameters(), lr=1e-4)

for epoch in range(3):
    for lr, hr in loader:
        fake_hr = G(lr)

        real_pred = D(hr)
        fake_pred = D(fake_hr.detach())

        d_loss = adversarial_loss(real_pred, torch.ones_like(real_pred)) + \
                 adversarial_loss(fake_pred, torch.zeros_like(fake_pred))

        opt_D.zero_grad()
        d_loss.backward()
        opt_D.step()

        fake_pred = D(fake_hr)
        perceptual = content_loss(vgg(fake_hr), vgg(hr))

        g_loss = content_loss(fake_hr, hr) + \
                 1e-3 * adversarial_loss(fake_pred, torch.ones_like(fake_pred)) + \
                 0.006 * perceptual

        opt_G.zero_grad()
        g_loss.backward()
        opt_G.step()

    print(f"SRGAN Epoch {epoch+1}")


## 5. Results and Comparison


In [ ]:
srcnn.eval()
G.eval()

lr, hr = dataset[0]

with torch.no_grad():
    srcnn_sr = srcnn(lr.unsqueeze(0)).squeeze(0)
    srgan_sr = G(lr.unsqueeze(0)).squeeze(0)

plt.figure(figsize=(12,4))
titles = ["Low-Res", "SRCNN", "SRGAN", "High-Res"]
images = [
    lr.permute(1,2,0),
    srcnn_sr.permute(1,2,0),
    srgan_sr.permute(1,2,0),
    hr.permute(1,2,0)
]

for i in range(4):
    plt.subplot(1,4,i+1)
    plt.imshow(images[i])
    plt.title(titles[i])
    plt.axis("off")

plt.show()
